# Воспроизведение baseline SASRec (Yandex YAMBDA)

Цель ноутбука — прогнать **оригинальный** код Яндекса (`yambda/benchmarks/models/sasrec/`) на Yambda-50M в режиме `listens` и убедиться, что воспроизводится цифра из таблицы `Listen+`:

- **NDCG@10 ≈ 0.0744**, **Recall@10 ≈ 0.0322** (см. `yandex_results/listen_results.png`).

Эта цифра — наш потолок для per-user скорера (gSASRec). Если наша реализация заметно ниже — значит дело в нашей тренировке/конфиге, а не в данных.

Запускать в **Google Colab (A100)**. Локально не имеет смысла — 100 эпох слишком долго.

## 1. Подтягиваем код Яндекса и ставим зависимости

Код бенчмарков лежит прямо в HF-датасете `yandex/yambda` под `benchmarks/` ([официальный README](https://huggingface.co/datasets/yandex/yambda/blob/main/benchmarks/README.md)). Качаем только нужную папку через `huggingface_hub.snapshot_download`.

In [ ]:
!pip install -q polars click datasets huggingface_hub tqdm

In [ ]:
from huggingface_hub import snapshot_download
import os

# Качаем только папку benchmarks/ из dataset-репозитория.
local_dir = snapshot_download(
    repo_id='yandex/yambda',
    repo_type='dataset',
    allow_patterns='benchmarks/*',
    local_dir='/content/yambda',
)
print('downloaded to:', local_dir)
print(os.listdir('/content/yambda/benchmarks'))

In [ ]:
%cd /content/yambda/benchmarks
# Ставим пакет 'yambda' (модули timesplit и т.п.) в editable режиме.
# Если pyproject подтянет лишние зависимости (implicit, sansa) — игнорируем
# и используем PYTHONPATH как fallback.
!pip install -q -e . || echo 'pip install failed; will use PYTHONPATH fallback'

os.environ['PYTHONPATH'] = '/content/yambda/benchmarks:' + os.environ.get('PYTHONPATH', '')
print('PYTHONPATH =', os.environ['PYTHONPATH'])

## 2. Скачиваем sequential/50m/listens.parquet

`train.py` ожидает структуру `data_dir/sequential/<size>/<interaction>.parquet` ([train.py:100](https://github.com/yandex-research/yambda/blob/main/benchmarks/models/sasrec/train.py#L100)).

In [ ]:
import os, shutil
from huggingface_hub import hf_hub_download

os.makedirs('/content/data/sequential/50m', exist_ok=True)
src = hf_hub_download(
    repo_id='yandex/yambda',
    filename='sequential/50m/listens.parquet',
    repo_type='dataset',
)
dst = '/content/data/sequential/50m/listens.parquet'
if not os.path.exists(dst):
    shutil.copy(src, dst)
print('size, MB:', os.path.getsize(dst) / 1024 / 1024)

## 3. Тренировка SASRec на Listen+

Дефолтный конфиг Яндекса:
- embedding_dim=64, num_heads=2, num_layers=2, dropout=0.0
- max_seq_len=200, batch_size=256, lr=1e-3
- **num_epochs=100**
- loss: plain BCE с 1 uniform-негативом на позицию

На A100 100 эпох на 50M ≈ 1.5–3 часа. Если сессия может оборваться — снизь `--num_epochs` до 30–50 как sanity-чек.

In [ ]:
!pip uninstall -y polars
!pip install -q --force-reinstall --no-cache-dir 'polars>=1.20'

In [ ]:
%cd /content/yambda/benchmarks/models/sasrec
!python train.py \
    --exp_name listens_50m \
    --data_dir /content/data \
    --checkpoint_dir /content/checkpoints \
    --size 50m \
    --interaction listens \
    --num_epochs 100 \
    --batch_size 256 \
    --device cuda:0

In [ ]:
import os, polars as pl, sys
from data import preprocess


sys.path.insert(0, '/content/yambda/benchmarks/models/sasrec')

p = '/content/data/sequential/50m/listens.parquet'
print('file size, GB:', os.path.getsize(p) / 1e9)

df_raw = pl.scan_parquet(p).collect(engine="streaming")
print('raw rows (users):', df_raw.height)
print('schema:', df_raw.schema)
print(df_raw.head(2))

data = preprocess(pl.scan_parquet(p), 'listens', val_size=0, max_seq_len=200)
train_df = data.train.collect(engine="streaming")
print('num_items:', data.num_items)
print('train rows after preprocess:', train_df.height)
print('avg seq len:', train_df.select(pl.col('item_id').list.len().mean()).item())


In [ ]:
# eval.py хардкодит путь ./checkpoints/{exp_name}_best_state.pth и не принимает --checkpoint_dir.
# Делаем симлинк, чтобы не копировать большой файл.

sasrec_dir = '/content/yambda/benchmarks/models/sasrec'
os.makedirs(f'{sasrec_dir}/checkpoints', exist_ok=True)
link = f'{sasrec_dir}/checkpoints/listens_50m_best_state.pth'
src = '/content/checkpoints/listens_50m_best_state.pth'
if not os.path.exists(link):
    os.symlink(src, link)

print('checkpoint linked:', os.path.exists(link))

## 4. Оценка (NDCG@10, Recall@10)

Финальные метрики печатает `eval.py`. Ожидаем числа в районе строки `SASRec` в таблице Listen+ для Yambda-50M:

| Метрика | Ожидание |
|---|---|
| NDCG@10 | 0.0744 |
| NDCG@100 | 0.0764 |
| Recall@10 | 0.0322 |
| Recall@100 | 0.1028 |

In [ ]:
%cd /content/yambda/benchmarks/models/sasrec
!python eval.py \
    --exp_name listens_50m \
    --data_dir /content/data \
    --size 50m \
    --interaction listens \
    --device cuda:0

## 5. Сохраняем чекпоинт в Google Drive (опционально)

Чтобы не терять обученный baseline между сессиями.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
out = '/content/drive/MyDrive/thesis/artifacts/yandex_sasrec_baseline'
os.makedirs(out, exist_ok=True)
shutil.copy('/content/checkpoints/listens_50m_best_state.pth', out)
print('saved to', out)

## Заметки

1. `train.py` сохраняет **последний** state, а не best-by-val ([train.py:144-145](https://github.com/yandex-research/yambda/blob/main/benchmarks/models/sasrec/train.py#L144-L145)). Промежуточные метрики не логирует — увидим только финальный `eval.py`.
2. Если хочется увидеть кривую обучения — нужно патчить `train.py` (вставить вызов `eval` после каждой эпохи). Для baseline-репродукции это излишне.
3. Запускали этот ноутбук — впиши в `docs/phase_1_log.md` фактические числа NDCG/Recall и расхождение с таблицей.